[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/fluids/convection/convection.ipynb)

# Rayleigh–Bénard Convection

This example computes Rayleigh–Bénard convection: a layer of liquid between a warm floor and a cool ceiling, which conducts heat up while it stays still and turns over in rolls once the warm liquid's buoyancy beats its viscosity and its conduction of heat. Warm liquid is lighter, so the still layer is top-heavy; a parcel that rises keeps its buoyancy only if it rises before its warmth has conducted away, and the Rayleigh number compares the two times. Here the layer turns over from a slightly disturbed, conducting start, and a sweep of Rayleigh numbers finds where a small disturbance stops dying and starts to grow, as [Bergé and Dubois (1984)](https://doi.org/10.1080/00107518408210730) set out.

The algebra is planar spacetime joined with two degenerate directions, one for volume and one for entropy, `Algebra("x+y+t-") * Algebra("v0s0")`. A parcel of the liquid is a point weighed by its volume, offset along time by its mass and along the entropy's direction by its entropy, `volume + (density - expansion * warmth) * time + warmth * entropy`: a unit volume holds a little less mass the warmer it is. A drive is minus the gradient of a potential wedged with what it acts on, and does its work on the flux of that quantity. Gravity's drive, wedged with time, pulls on the mass a flow carries, and since the still layer's weight is held by the pressure, only a warm parcel's missing mass is left to pull on. The temperature's drive, its gradient wedged with entropy, does no work on the flow: a conductivity maps it to the entropy's flux, which leaves each cell as the entropy the flow carries does. The flow is a stream function on the mesh's vertices; its mass resists its change and its viscosity its turning, and the pressure, which only keeps the flow from filling or emptying any cell, does no work on it and drops out. The momentum's own advection is left out, as the large Prandtl number of water and oils makes it small.

In vector-calculus notation, the Oberbeck–Boussinesq equations read $\partial_t\mathbf v = -\nabla p/\rho_0 + \nu\nabla^2\mathbf v + g\alpha T\,\hat{\mathbf z}$, $\nabla\cdot\mathbf v = 0$ and $\partial_t T + \mathbf v\cdot\nabla T = \kappa\nabla^2 T$, here without the term $(\mathbf v\cdot\nabla)\mathbf v$; the Rayleigh number is $\mathrm{Ra} = g\alpha\,\Delta T\,d^3/(\nu\kappa)$, and between rigid plates the still layer gives way at $\mathrm{Ra}_c \approx 1707$, to rolls of wavenumber $a_c \approx 3.117$ times the inverse depth.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from itertools import islice, tee

import numpy as np

from numga import Algebra, NumpyContext, stack
from numga.sparse import spdiag
from examples.util.animation import show_animation
from examples.fluids.convection import render, scenarios
from examples.fluids.convection.core import layer
from examples.util.topology import Topology

# Planar spacetime, and the volume and the entropy a parcel holds: each a direction of its own, degenerate, since
# nothing measures a volume or an entropy against a length or a time.
spacetime = Algebra("x+y+t-")
ga = spacetime * Algebra("v0s0")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
Point = ga.gatype.antivector()                                                # Antivector-
Plane = ga.gatype.vector()                                                    # Vector+
# The planes reading a place's run along the layer and up it.
Upright = ga.gatype.from_blades("x y")                                        # Vector+
Drive = ga.gatype.from_blades("xt yt xv yv xs ys")                            # Bivector+
# The layer's two directions, along its length and up its depth.
length = mv.x.dual()                                                          # [] Point-
height = mv.y.dual()                                                          # [] Point-
time = -mv.t.dual()                                                           # [] Point-
# A unit of volume, the weight of a place, and a unit of entropy.
volume = mv.v.dual()                                                          # [] Point-
entropy = mv.s.dual()                                                         # [] Point-

## The layer

The layer is a strip of triangles one depth high, closed on itself along its length so that it has a floor and a ceiling but no ends. Its geometry is read on the same strip opened at its seam, its last column the first again a period further along, so that no edge needs telling that it crosses the seam: the two share their triangles, in the same order, and a field on the closed layer reads on the open one. Lengths are in depths, times in the time heat takes to conduct across the layer, and temperatures in the difference between floor and ceiling, a degree.

In [ ]:
def triangles(topology: Topology, vertices: Point) -> tuple[Scalar, Point, Plane]:
    """Each triangle's area `Scalar+[faces]`, the side facing each corner `[corners] Point-[faces]`, and each corner's
    hat gradient `[corners] Plane+[faces]`, which reads one from the facing side to the corner."""
    sides = topology.boundary[2, 1] * (topology.d[0] * vertices)              # [corners] Point-[faces]
    # Twice a triangle's area: two of its sides joined, read through time, at unit volume and entropy.
    double_areas = (((sides[0] & sides[1]) & time) & volume) & entropy        # Scalar-[faces]
    # A corner's hat gradient: the plane through the facing side, along time, volume and entropy, over twice the area.
    gradients = (time & volume & entropy & sides) / double_areas              # [corners] Plane+[faces]
    return double_areas.norm() / 2, sides, gradients


# The layer, four depths long, closed and opened at its seam, the maps reading a field on each at the other's vertices,
# and the open layer's vertices, places of unit volume.
closed, opened, closing, opening, places = layer(scenarios.n_along, scenarios.n_across, scenarios.period)
vertices = (volume + mv(Upright, places).dual()).field()                      # Point-[opened vertices]
areas, sides, gradients = triangles(opened, vertices)
render.draw_layer(opened, vertices, scenarios.period);

The layer's triangles lie between its warm floor, in red, and its cool ceiling, in blue.

## Turning over

The liquid's flow is a stream function, whose rise at a corner carries the liquid along the facing side. Its kinetic energy is the momentum paired with the velocity, and its conduction of heat is the outflow of the entropy flux the temperature's drive conducts, `mv.s & (gradients ^ conductivity(drives))`; over a triangle the two give the same numbers, the one as a flow's energy, the other as a flux. Gravity's drive does work on the flows the stream function can make by pulling on the mass they carry. The liquid has water's Prandtl number, its viscosity seven times its conduction of heat, and the layer starts still and conducting, with a small disturbance. The cells below take a few seconds, most of it stepping the liquid and drawing the animation.

In [ ]:
# The liquid: its density, its viscosity, the Prandtl number in these units, the mass a unit volume loses per degree,
# and gravity, which with them makes the Rayleigh number.
density, viscosity, expansion = scenarios.density, scenarios.prandtl, scenarios.expansion
gravity = scenarios.rayleigh * viscosity / expansion

In [ ]:
def convecting(topology: Topology, areas: Scalar, sides: Point, gradients: Plane, gravity: np.ndarray,
               temperature: Scalar, dt: float, n_steps: int, n_stride: int) -> Iterator[tuple[Scalar, Scalar]]:
    """The layer's stream function and temperature `[...] Scalar+[vertices]` every `n_stride` steps of `dt`, from rest
    and this temperature, under each gravity `[...]`, the stream function nothing and the temperature held at the
    floor and the ceiling: each step gravity's drive does work on the flows the stream function can make, the
    liquid's mass and viscosity resist them, and the flow carries the entropy along while the temperature's drive
    conducts it."""
    corners = topology.incidence[2, 0]                                        # [corners] Scalar+[faces <- vertices]
    interior = topology.reading(topology.interior[0])                         # Scalar+[free <- vertices]
    # Each vertex's share of the area around it.
    cells = (~corners * (areas / 3)).sum(axis=0)                              # Scalar+[vertices]
    # A stream function rising by one at a corner carries the liquid along the facing side, half the side's worth over
    # the triangle: each corner's flow, and its velocity.
    flows = sides / 2                                                         # [corners] Point-[faces]
    velocities = flows / areas                                                # [corners] Point-[faces]
    # The kinetic form: the liquid's inertia reads a velocity's run along and up into its momentum, paired with the
    # velocity of each other corner over the triangle, and gathered onto the vertices.
    inertia = density * (stack((mv.x, mv.y)) * (stack((mv.x, mv.y)) & Point)).sum(axis=0)   # [] Plane+ <- Point-
    local = areas * (inertia(velocities[:, None]) & velocities[None, :])      # [corners, corners] Scalar+[faces]
    kinetic = (~corners[:, None] * spdiag(local) * corners[None, :]).sum(axis=0).sum(axis=0)   # Scalar+[vertices <- vertices]
    # The vorticity at each vertex, the kinetic form's amount over its share of the area and the density, and the
    # viscous form, its square over the layer.
    vorticity = spdiag(1 / (density * cells)) * kinetic                       # Scalar+[vertices <- vertices]
    viscous = viscosity * ~vorticity * spdiag(cells) * vorticity              # Scalar+[vertices <- vertices]
    # The temperature's drive at each corner's hat, minus its gradient wedged with the entropy it pushes, and the
    # conductivity, the drive read across each of the layer's directions driving the entropy along it.
    drives = -(gradients ^ mv.s)                                              # [corners] Drive+[faces]
    swept = stack((length, height)) & entropy                                 # [2] Flux-
    conductivity = (swept * (swept & Drive)).sum(axis=0)                      # [] Flux- <- Drive+
    # What a flux carries out of each corner's cell: its hat gradient wedged with the flux, read for its entropy.
    local = areas * (mv.s & (gradients[:, None] ^ conductivity(drives[None, :])))   # [corners, corners] Scalar+[faces]
    conduction = (~corners[:, None] * spdiag(local) * corners[None, :]).sum(axis=0).sum(axis=0)   # Scalar+[vertices <- vertices]
    # Each step's implicit maps on the free vertices: the flow's inertia with its viscosity, and the cells' heat with
    # its conduction.
    flow_step = (interior * (kinetic / dt + viscous) * ~interior) * Scalar   # (Scalar+ <- Scalar+)[free <- free]
    heat_step = (interior * (spdiag(cells / dt) + conduction) * ~interior) * Scalar   # (Scalar+ <- Scalar+)[free <- free]
    pull = -(gravity * mv.yt)                                                 # [...] Drive+
    stream = 0 * temperature                                                  # [...] Scalar+[vertices]
    for step in range(n_steps + 1):
        if step % n_stride == 0:
            yield stream, temperature
        # Each triangle's parcels per volume: unit volume, holding less mass the warmer they are, and their entropy.
        warmth = corners.mean(axis=0) * temperature                           # [...] Scalar+[faces]
        parcels = volume + (density - expansion * warmth) * time + warmth * entropy   # [...] Point-[faces]
        load = (~corners * (pull[..., None] & (flows & parcels[..., None]))).sum(axis=-1)   # [...] Scalar+[vertices]
        stream = ~interior * flow_step.solve(interior * (kinetic / dt * stream + load))
        # The flow carries the entropy along; what its flux carries out of each corner's cell leaves it, and the
        # temperature's drive conducts the rest, implicitly.
        velocity = ((corners * stream[..., None]) * velocities).sum(axis=-1)   # [...] Point-[faces]
        flux = velocity & (warmth * entropy)                                  # [...] Flux-[faces]
        advected = mv.s & (~corners * (areas * (gradients ^ flux[..., None]))).sum(axis=-1)   # [...] Scalar+[vertices]
        wall_temperature = temperature - ~interior * (interior * temperature)   # [...] Scalar+[vertices]
        temperature = wall_temperature + ~interior * heat_step.solve(
            interior * (cells / dt * temperature - advected - conduction * wall_temperature))

In [ ]:
start = scenarios.conducting(closing * (mv.y & vertices), scenarios.disturbance)   # Scalar+[vertices]
# Shown once the rolls begin to show, each frame as it comes, and kept for the checks.
animation_states, frames = tee(islice(convecting(closed, areas, sides, gradients, np.array(gravity), start, scenarios.dt,
                                                 scenarios.n_steps, scenarios.n_stride),
                                      scenarios.n_unshown // scenarios.n_stride, None))
show_animation(render.animate(opened, vertices, opening, scenarios.period, animation_states), "convection",
               scenarios.frame_duration_ms, n_frames=(scenarios.n_steps - scenarios.n_unshown) // scenarios.n_stride + 1)

The layer is coloured by its temperature, red above the middle's and blue below, with the streamlines in black at a fixed spacing of the stream function, so that the closer they lie the faster the liquid flows. The animation starts as the disturbance's rolls begin to show in the streamlines, while the layer still conducts. The rolls grow, warm liquid rising and cool liquid sinking between them, and overshoot: they turn the warm liquid right over, mixing the layer's middle until the top-heaviness that drives them is spent, and nearly stall. Conduction builds the warm layer at the floor and the cool one at the ceiling up again, the rolls pick up, and after a few swings they carry the heat up steadily; the temperature then changes only in thin layers at the floor and the ceiling, where conduction hands the heat to the flow and takes it back.

## The onset

A small disturbance of the still layer grows or dies at a steady rate once its other parts have died away, and the rate's sign tells whether the layer is stable. The layer runs one wavelength of the first rolls long, at several Rayleigh numbers at once, a batch of gravities, and the growth rate is read from the stream function's size at two times. This cell takes about five seconds.

In [ ]:
# The layer one wavelength of the first rolls long, at these Rayleigh numbers.
rayleighs = scenarios.onset_rayleighs                                         # [rayleighs]
onset_closed, onset_opened, onset_closing, _, onset_places = layer(scenarios.n_onset_along, scenarios.n_across,
                                                                   scenarios.onset_period)
onset_vertices = (volume + mv(Upright, onset_places).dual()).field()          # Point-[opened vertices]
onset_areas, onset_sides, onset_gradients = triangles(onset_opened, onset_vertices)
# The still layer, disturbed by a roll a wavelength along and half a wave up.
heights = onset_closing * (mv.y & onset_vertices)                             # Scalar+[vertices]
runs = onset_closing * (mv.x & onset_vertices)                                # Scalar+[vertices]
rolls = scenarios.rolls_size * (scenarios.critical_wavenumber * runs).sin() * (np.pi * heights).sin()   # Scalar+[vertices]
readings = list(convecting(onset_closed, onset_areas, onset_sides, onset_gradients, rayleighs * viscosity / expansion,
                           scenarios.conducting(heights, 0.0) + rolls, scenarios.onset_dt, scenarios.n_onset_steps,
                           scenarios.n_onset_stride))
# The stream function's squared size over the layer at the middle and at the end; it grows at twice the rate.
onset_cells = (~onset_closed.incidence[2, 0] * (onset_areas / 3)).sum(axis=0)   # Scalar+[vertices]
sizes = [(onset_cells * stream.squared()).sites.sum() for stream, _ in readings[1:]]   # [readings] [rayleighs] Scalar+
growth = (sizes[1].log() - sizes[0].log()) / (2 * scenarios.n_onset_stride * scenarios.onset_dt)   # [rayleighs] Scalar+
render.draw_onset(rayleighs, growth);

The growth rate against the Rayleigh number crosses zero at the dotted line, the onset between rigid plates: below it a disturbance dies, above it the rolls grow.

In [ ]:
# checks
# Gravity pulls a parcel's mass down and passes over its volume and its entropy.
pull_on_parcel = (mv.y & (-mv.yt).commutator(time + volume + entropy)).to_array()
np.testing.assert_allclose(pull_on_parcel, -1.0, rtol=1e-12)
# The growth rate crosses zero within one percent of the onset between rigid plates.
rates = growth.to_array()
crossing = np.interp(0.0, rates, rayleighs)
np.testing.assert_allclose(crossing, 1707.8, rtol=0.01)
# Turned over, the layer's middle half is all but even in temperature on average, the drop in the thin layers at the
# floor and the ceiling; conducting, it drops half a degree.
layer_heights = (mv.y & vertices).to_array()
temperatures = (opening * list(frames)[-1][1]).to_array()
middle_drop = temperatures[np.isclose(layer_heights, 0.25)].mean() - temperatures[np.isclose(layer_heights, 0.75)].mean()
assert abs(middle_drop) < 0.05